In [2]:
import os
import pandas as pd
import numpy as np
import json
from PIL import Image
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from tqdm import tqdm
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ==========================================
# 1. CONFIGURATION
# ==========================================
# Ensure this path matches your Kaggle input sidebar
SARTAJ_PATH = Path("/kaggle/input/datasets/sartajbhuvaji/brain-tumor-classification-mri") 
HF_REPO_ID = "srijan2715/sartaj-final-split" # New clean repo
SEED = 42

# Label mapping (handling the 'notumor' folder name)
label_to_int = {"glioma": 0, "meningioma": 1, "no": 2, "pituitary": 3}

# ==========================================
# 2. SCAN RAW FOLDERS (NO CLEANING, EXACT PAPER MATCH)
# ==========================================
print("🔍 SCANNING OFFICIAL KAGGLE FOLDERS...")
records = []

for split_name in ['Training', 'Testing']:
    split_path = SARTAJ_PATH / split_name
    if not split_path.exists(): 
        print(f"❌ Path not found: {split_path}")
        continue
        
    for class_dir in split_path.iterdir():
        if not class_dir.is_dir(): continue
        
        # Normalize folder names: "no_tumor" or "notumor" -> "no"
        class_name = class_dir.name.lower().replace("_tumor", "").replace("notumor", "no")
        
        for img_path in class_dir.glob("*.*"):
            if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                records.append({
                    "split": "train" if split_name == "Training" else "test",
                    "image_path": str(img_path),
                    "label_str": class_name
                })

df = pd.DataFrame(records)
print(f"📊 TOTAL RAW IMAGES FOUND: {len(df)} (Should be ~3264)")
print(df['label_str'].value_counts())

# ==========================================
# 3. SPLIT DATA (Official Test Set + 85/15 Train/Val Split)
# ==========================================
train_df = df[df['split'] == 'train'].copy()
test_df = df[df['split'] == 'test'].copy()

# Create validation set from training data (Standard practice for checkpointing)
train_split, val_split = train_test_split(
    train_df, test_size=0.15, 
    stratify=train_df['label_str'], 
    random_state=SEED
)

print(f"\n📐 FINAL SPLITS: Train={len(train_split)}, Val={len(val_split)}, Test={len(test_df)}")
print(f"📐 TOTAL: {len(train_split) + len(val_split) + len(test_df)}")

# ==========================================
# 4. PREPROCESS (PAPER EXACT MATCH: 224x224)
# ==========================================
def process_images(df, split_name):
    images, labels_int = [], []
    failed_count = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        if row['label_str'] not in label_to_int:
            failed_count += 1
            continue
            
        try:
            img = Image.open(row['image_path']).convert("RGB")
            # PAPER EXACT MATCH: Direct resize to 224x224
            img_final = img.resize((224, 224), Image.Resampling.BICUBIC)
            
            images.append(np.array(img_final))
            labels_int.append(label_to_int[row['label_str']])
        except Exception as e:
            failed_count += 1
            
    print(f"Failed/skipped in {split_name}: {failed_count}")
    return images, labels_int

print("\n🖼️ RESIZING TO 224x224...")
train_imgs, train_lbls = process_images(train_split, "train")
val_imgs, val_lbls = process_images(val_split, "val")
test_imgs, test_lbls = process_images(test_df, "test")

# ==========================================
# 5. UPLOAD
# ==========================================
print("\n🚀 UPLOADING TO HUGGING FACE...")
dataset_dict = DatasetDict({
    "train": Dataset.from_dict({"image": train_imgs, "label": train_lbls}),
    "val": Dataset.from_dict({"image": val_imgs, "label": val_lbls}),
    "test": Dataset.from_dict({"image": test_imgs, "label": test_lbls})
})

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

dataset_dict.push_to_hub(HF_REPO_ID, token=hf_token, private=False, max_shard_size="500MB")
print(f"\n🎉 DONE! Link: https://huggingface.co/datasets/{HF_REPO_ID}")

🔍 SCANNING OFFICIAL KAGGLE FOLDERS...
📊 TOTAL RAW IMAGES FOUND: 3264 (Should be ~3264)
label_str
meningioma    937
glioma        926
pituitary     901
no            500
Name: count, dtype: int64

📐 FINAL SPLITS: Train=2439, Val=431, Test=394
📐 TOTAL: 3264

🖼️ RESIZING TO 224x224...


Processing train: 100%|██████████| 2439/2439 [00:20<00:00, 121.50it/s]


Failed/skipped in train: 0


Processing val: 100%|██████████| 431/431 [00:03<00:00, 123.12it/s]


Failed/skipped in val: 0


...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


🎉 DONE! Link: https://huggingface.co/datasets/srijan2715/sartaj-final-split


In [3]:
import json

label_to_int = {"glioma": 0, "meningioma": 1, "no": 2, "pituitary": 3}

# Save it to the current working directory
with open("label_mapping.json", "w") as f:
    json.dump(label_to_int, f, indent=2)

print("✅ label_mapping.json created successfully!")

✅ label_mapping.json created successfully!
